# 01.3 — The stack

People describe RAG as one pipeline. It's two, and they run at completely
different times, on different hardware, with different failure modes.

Getting this distinction straight now saves a lot of confusion later, because
almost every design decision in the course is really a question of which
pipeline pays for it.

In [1]:
!pip install -q pymupdf4llm==1.28.2

## Pipeline one: ingestion

Runs when documents arrive. Turns files into something searchable.

Note the naming: **ingestion** is the whole pipeline; **indexing** is only its last
stage. Keeping those apart matters, because most of the work — and most of the
failures — happen in the stages before the index is ever touched.

```
  documents  →  parse  →  chunk  →  embed  →  index
```

Watch the first two stages happen, and time them.

In [2]:
import time
from pathlib import Path
import pymupdf4llm

CORPUS = Path('../../corpus/docs')
NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]

start = time.perf_counter()
docs = {n: pymupdf4llm.to_markdown(str(CORPUS / n)) for n in NATIVE_PDFS}
parse_seconds = time.perf_counter() - start

start = time.perf_counter()
chunks = [t[i:i + 500] for t in docs.values() for i in range(0, len(t), 500)]
chunk_seconds = time.perf_counter() - start

print(f'parse {len(docs)} documents : {parse_seconds * 1000:8.0f} ms')
print(f'chunk into {len(chunks)}      : {chunk_seconds * 1000:8.0f} ms')
print(f'\nper document          : {parse_seconds / len(docs) * 1000:.0f} ms')
print(f'10,000 documents      : {parse_seconds / len(docs) * 10_000 / 60:.0f} minutes, parsing alone')

parse 7 documents :     2782 ms
chunk into 71      :        0 ms

per document          : 397 ms
10,000 documents      : 66 minutes, parsing alone


Parsing takes hundreds of milliseconds per document. Chunking is free by
comparison — it's string slicing.

At seven documents, nobody cares. At ten thousand, parsing alone is a job you
schedule rather than a function you call, and that's before embedding. This is
why ingestion is a **batch pipeline**: it runs offline, it's allowed to be slow,
and it needs to survive being interrupted halfway through.

Add embedding and the picture gets worse — every chunk goes through a model.
You'll do that in module 02.

## Pipeline two: querying

Runs on every request, while somebody waits.

```
  question  →  embed  →  search  →  assemble context  →  generate  →  check  →  answer
```

Same word, `embed`, in both pipelines — and it must be the same model in both, or
the numbers aren't comparable and nothing matches anything. That's a real bug
people ship.

Everything here happens inside a few seconds, and the budget is tight. Retrieval
should stay under roughly 300 milliseconds; generation runs two to three seconds
for a small model and five to ten for a large one. Every component you add to
this pipeline is spending someone's attention.

## The last stage is the one people leave out

`check` in the diagram above. After the model writes an answer and before the
user sees it:

- is the answer actually supported by the retrieved text, or did the model
  improvise?
- does it contain something it shouldn't — personal data, a policy breach?
- did a retrieved document contain instructions aimed at the model rather than
  content aimed at the reader?

Most architecture diagrams stop at `generate`. Drawing the check stage in from
the beginning is a decision about whether you'll ever build it. Modules 11 and
16.

## Vocabulary

Used precisely from here on, because three of these get muddled constantly.

| Term | Meaning |
| --- | --- |
| **Document** | A file as it arrived. `sahel-procurement-policy-v3.pdf` |
| **Corpus** | The whole set of documents. Our 15 files. |
| **Chunk** | A piece of a document, small enough to embed and retrieve |
| **Embedding** | A chunk turned into a list of numbers positioned by meaning |
| **Index** | The searchable structure holding those numbers |
| **Ingestion** | The whole offline pipeline: parse, chunk, embed, index |
| **Retrieval** | Finding the chunks most likely to answer a question |
| **Context** | The retrieved chunks, formatted into the prompt |
| **Generation** | The model writing an answer from that context |

The three that get confused:

**Corpus is not index.** The corpus is your documents. The index is a derived
artifact you can throw away and rebuild. Changing your chunking strategy means
rebuilding the index — the corpus doesn't move.

**Retrieval is not context.** Retrieval returns chunks. Context is what you
actually put in the prompt after deciding how many, in what order, in what
format. Module 10 is entirely about that gap.

**A chunk is not a document.** Obvious until you're writing a citation and
realise your chunk no longer knows which file it came from.

**Ingestion is not indexing.** Ingestion is the whole offline pipeline; indexing
is its final stage. Worth keeping straight, because almost everything that goes
wrong offline goes wrong before the index is touched — and module 03 is named
for the pipeline, not the stage.

## Both pipelines, together

```
  INGESTION  (offline, batch, slow, runs when documents change)

      documents ──▶ parse ──▶ chunk ──▶ embed ──▶ index
                                           │         │
                                    same model!      │
                                           │         ▼
  QUERYING  (online, per request, fast)    │     ┌───────┐
                                           │     │ index │
      question ──▶ embed ─────────────────┘     └───┬───┘
                     │                               │
                     └────────▶ search ◀─────────────┘
                                  │
                                  ▼
                          assemble context
                                  │
                                  ▼
                             generate
                                  │
                                  ▼
                               check
                                  │
                                  ▼
                               answer
```

Two things to take from it. The index is the only thing the pipelines share, and
the embedding model appears in both — which is why swapping it means re-indexing
everything, and why that decision is more expensive than it looks.

## What's next

You know the stages. Notebook 4 shows you what each one does when it breaks —
using the results from a pipeline that has already run, so the failures are
measured rather than imagined.